# 7-1절 연습 문제 풀이

이 노트북은 7-1절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch07/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

## 연습 7-1

자연어를 처리하는 모델을 설계 중이다. 단어를 토큰으로 했을 때 어휘 사전을 구성하는 고유 토큰의 수가 1만 개라고 가정하자. 이 토큰을 원-핫 인코딩으로 표현하는 방식과 임베딩 차원이 128인 임베딩 벡터로 표현하는 방식을 다음 두 관점에서 비교해 보자.

메모리 및 연산 효율성

단어 간 관계의 표현 능력

In [ ]:
VOCAB, EMBED = 10000, 128
onehot_bytes = VOCAB * VOCAB * 4        # 원-핫 행렬을 모두 저장한다면
embed_bytes = VOCAB * EMBED * 4         # 임베딩 행렬
print(f'원-핫 표현 한 토큰: {VOCAB:,}차원 (대부분 0)')
print(f'임베딩 표현 한 토큰: {EMBED}차원')
print(f'다음 계층이 은닉 256일 때 가중치 수')
print(f'  원-핫 입력: {VOCAB * 256:,}개')
print(f'  임베딩 입력: {VOCAB * EMBED + EMBED * 256:,}개 (임베딩 행렬 포함)')

**메모리와 연산 효율**
원-핫은 토큰 하나가 10,000차원이고 그중 하나만 1이라 낭비가 크다. 임베딩은 128차원으로 줄어 다음 계층의 가중치도 크게 줄고, 조회(lookup) 연산이라 행렬 곱보다 빠르다.

**단어 간 관계의 표현 능력**
원-핫 벡터는 서로 직교해서 어떤 두 단어의 내적도 0이다. 즉 '고양이'와 '개'가 '고양이'와 '자동차'만큼 멀다. 임베딩은 학습을 통해 **비슷한 맥락에 쓰이는 단어를 가까운 위치**에 놓으므로 단어 사이의 유사도를 표현할 수 있다.

## 연습 7-2

[연습 문제 7-1]의 자연어 모델의 목적을 하나 설정하고, 그 목적을 기준으로 임베딩 행렬이 만들어지는 과정을 설명해 보자. 그리고 그 결과로 임베딩 행렬의 가중치가 어떤 특성을 갖게 될지 이야기해 보자.

### 풀이

**목적 설정**: 다음 단어를 예측하는 언어 모델(7-2절의 OzWriter와 같은 과제)로 정한다.

**임베딩 행렬이 만들어지는 과정**
1. 임베딩 행렬은 (어휘 수 10,000 × 임베딩 차원 128) 크기의 **학습 대상 파라미터**로, 처음에는 무작위 값이다.
2. 입력 토큰의 인덱스에 해당하는 **행 하나**를 꺼내 다음 계층으로 보낸다.
3. 예측이 틀리면 손실이 커지고, 역전파로 그 행의 값이 조정된다.
4. 이 과정이 반복되면 **비슷한 예측 결과를 내야 하는 단어들의 행이 서로 비슷해진다**.

**결과로 얻는 특성**
같은 자리에 바꿔 쓸 수 있는 단어(예: king, queen 또는 월요일, 화요일)는 벡터가 가까워진다. 목적이 달라지면 임베딩의 성격도 달라진다. 감성 분류로 학습하면 '좋다'와 '훌륭하다'가 가까워지고, 품사 태깅으로 학습하면 같은 품사끼리 모인다. 즉 **임베딩은 과제가 정의하는 의미를 담는다**.

## 연습 7-3

텍스트 데이터의 임베딩은 보통 글자를 토큰으로 사용하지 않고 단어 또는 어절(한국어 텍스트의 경우)을 토큰으로 사용한다. 그 이유가 무엇일까? 그리고 어떤 상황에서 평소와 달리 글자를 토큰으로 사용할까?

### 풀이

**단어·어절을 토큰으로 쓰는 이유**
의미의 최소 단위가 단어이기 때문이다. 글자 단위로 쪼개면 '사'라는 글자가 '사과', '회사', '사랑'에서 전혀 다른 뜻으로 쓰여 하나의 벡터로 표현하기 어렵다. 또 같은 문장을 표현하는 데 토큰 수가 훨씬 많아져 순환 신경망이 먼 거리의 관계를 학습하기 힘들어진다.

**글자를 토큰으로 쓰는 상황**
1. **어휘가 폭발하는 경우**: 한국어처럼 조사가 붙어 어절 종류가 매우 많거나, 신조어·오타가 많은 데이터
2. **처음 보는 단어를 다뤄야 할 때**: 글자 조합으로 어떤 단어든 표현할 수 있다
3. **글자 자체가 과제인 경우**: 6-3절의 띄어쓰기 모델처럼 글자 위치를 판단하는 과제
4. 문자 수가 적은 언어(영어 알파벳 등)

실무에서는 둘의 절충인 **서브워드 토큰화**(BPE 등)를 주로 쓰며, 12장의 LLM 토크나이저가 이 방식이다.